In [ ]:
import pandas as pd
import requests
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed


FDC_API_KEY = os.getenv("FDC_API_KEY")

synonyms = {
    'venison': 'deer',
    'soup/stew': 'soup',
}
to_mg = {'g': 1000, 'mg': 1, 'mcg': 0.001, "ug": 0.001, "µg": 0.001}
valid_values = {
    # компоненты еды
    1004: ("Fat", "g", 78),
    1258: ("Saturated fat", "g", 20),
    1253: ("Cholesterol", "mg", 300),
    1005: ("Total carbohydrates", "g", 275),
    1093: ("Sodium", "mg", 2300),
    1079: ("Dietary Fiber", "g", 28),
    1003: ("Protein", "g", 50),
    1235: ("Added sugars", "g", 50),

    # витаминчики :)
    1106: ("Vitamin A", "mcg", 900),      
    1162: ("Vitamin C", "mg", 90),
    1087: ("Calcium", "mg", 1300),
    1089: ("Iron", "mg", 18),
    1114: ("Vitamin D", "mcg", 20),      
    1109: ("Vitamin E", "mg", 15),
    1185: ("Vitamin K", "mcg", 120),
    1165: ("Thiamin", "mg", 1.2),
    1166: ("Riboflavin", "mg", 1.3),
    1167: ("Niacin", "mg", 16),
    1175: ("Vitamin B6", "mg", 1.7),
    1190: ("Folate", "mcg", 400),          
    1178: ("Vitamin B12", "mcg", 2.4),
    1176: ("Biotin", "mcg", 30),
    1170: ("Pantothenic acid", "mg", 5),
    1091: ("Phosphorus", "mg", 1250),
    1100: ("Iodine", "mcg", 150),
    1090: ("Magnesium", "mg", 420),
    1095: ("Zinc", "mg", 11),
    1103: ("Selenium", "mcg", 55),
    1098: ("Copper", "mg", 0.9),
    1101: ("Manganese", "mg", 2.3),
    1096: ("Chromium", "mcg", 35),
    1102: ("Molybdenum", "mcg", 45),
    1088: ("Chloride", "mg", 2300),
    1092: ("Potassium", "mg", 4700),
    1180: ("Choline", "mg", 550),
}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
    'Accept': 'application/json',
    'Content-Type': 'application/json',
}

def fetch(query):
    if query in cache:
        return query, cache[query], 200
    response = session.get("https://api.nal.usda.gov/fdc/v1/foods/search", params={
        'query': query,
        'api_key': FDC_API_KEY,
        'pageSize': 50,
    }, timeout=10)
    if response.status_code != 200:
        return query, None, response.status_code
    return query, response.json(), 200

def to_percent(nutrient_id, amount, unit):
    something, daily_value_unit, daily_value = valid_values[nutrient_id]
    unit = unit.lower()
    if unit not in to_mg:
        return None
    amount = amount * to_mg[unit] / to_mg[daily_value_unit]
    return round(amount / daily_value * 100, 1)

def right_food(item, data):
    item = item.lower().strip()
    needed_heads = {item, item + 's', item + 'es'}
    if item.endswith('y'):
        needed_heads.add(item[:-1] + 'ies')

    candidates = []
    for food in data.get('foods', []):
        desc = food.get('description', '').lower()
        parts = [part.strip() for part in desc.split(',')]

        if parts[0] in needed_heads:
            score = 3                                  
        elif any(part in needed_heads or part.startswith(item + ' ') for part in parts):
            score = 1
        else:                                 
            continue

        if 'raw' in desc:
            score += 2
        if food.get('dataType') == 'Foundation':
            score += 1
        score -= desc.count(',') * 0.1
        candidates.append((score, food))

    if not candidates:
        return None
    return max(candidates, key=lambda x: x[0])[1]
    
try:
    with open('data/ingredients_cache.json', encoding='utf-8') as file:
        cache = json.load(file)
except FileNotFoundError:
    cache = {}

session = requests.Session()

df_nutrients = pd.read_csv('data/ingredient_cols.csv', index_col='Unnamed: 0')
df_nutrients = df_nutrients.rename(columns={'0': 'ingredient'})

items = df_nutrients['ingredient'].dropna().unique()
queries = {item: synonyms.get(item, item).replace('/', ' ') for item in items}

responses = {}
errors = {}
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(fetch, query) for query in set(queries.values())]
    for future in as_completed(futures):
        query, data, status = future.result()
        if status is None:
            errors[query] = status
        else:
            responses[query] = data
            cache[query] = data

with open('data/ingredients_cache.json', 'w', encoding='utf-8') as file:
    json.dump(cache, file)

rows = []
not_found = []

for item, query in queries.items():
    if query in errors:
        not_found.append({'food': item, 'reason': f'{errors[query]}'})
        continue

    food = right_food(query, responses[query])
    if food is None:
        not_found.append({'food': item, 'reason': 'food is None'})
        continue

    row = {'ingredient': item, 'description': food.get('description')}
    for nutrient in food.get('foodNutrients', []):
        nutrient_id = nutrient.get('nutrientId')
        if nutrient_id not in valid_values:
            continue
        name = valid_values[nutrient_id][0]
        row[name] = to_percent(
            nutrient_id, nutrient.get('value', 0), nutrient.get('unitName', '')
        )
    rows.append(row)

df_result = pd.DataFrame(rows)
df_not_found = pd.DataFrame(not_found)
df_result   

,ingredient,description,Vitamin A,Folate,Vitamin E,Choline,Total carbohydrates,Fat,Niacin,Pantothenic acid,...,Manganese,Saturated fat,Cholesterol,Vitamin D,Selenium,Vitamin K,Biotin,Iodine,Added sugars,Molybdenum
0,lima bean,"Lima beans, immature seeds, raw",1.1,8.5,2.1,7.3,7.3,1.1,9.2,4.9,...,53.0,1.0,0.0,0.0,3.3,4.7,NaN,NaN,NaN,NaN
1,orzo,ORZO,NaN,NaN,NaN,NaN,28.6,2.3,33.5,NaN,...,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,banana,"Bananas, overripe, raw",0.1,NaN,NaN,NaN,7.3,0.3,3.6,NaN,...,NaN,NaN,NaN,NaN,NaN,0.2,0.0,NaN,NaN,NaN
3,tilapia,"Fish, tilapia, farm raised, raw",NaN,NaN,NaN,NaN,0.0,3.2,NaN,NaN,...,0.2,3.2,15.9,NaN,37.3,NaN,NaN,2.8,NaN,NaN
4,berry,"Berries, frozen",0.3,5.0,4.7,1.4,4.5,0.4,4.4,NaN,...,NaN,0.1,0.0,0.0,0.7,9.7,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
302,amaretto,"AMARETTO, IRISH CREAM, RUM HOT CHOCOLATE BOMBE...",NaN,NaN,NaN,NaN,20.4,41.0,NaN,NaN,...,NaN,100.0,2.0,NaN,NaN,NaN,NaN,NaN,84.0,NaN
303,potato salad,POTATO SALAD,NaN,NaN,NaN,NaN,7.8,11.0,NaN,NaN,...,NaN,7.1,6.0,NaN,NaN,NaN,NaN,NaN,14.2,NaN
304,poppy,Poppy seed dressing,5.7,0.8,15.7,1.1,8.6,42.7,0.3,NaN,...,NaN,30.3,5.0,0.5,2.2,41.8,NaN,NaN,NaN,NaN
305,kiwi,KIWI,NaN,NaN,NaN,NaN,5.4,0.9,NaN,NaN,...,NaN,0.0,0.0,NaN,NaN,31.5,NaN,NaN,NaN,NaN


In [23]:
df_merged = df_nutrients.merge(df_result, on='ingredient', how='left').fillna(0).drop(columns=['description'])
df_merged.to_csv('data/nutrition_facts.csv', index=False)

# df_result[df_result['ingredient'].isin(df_not_found['food'])]